<a href="https://colab.research.google.com/github/Nancy-Chauhan/Evals_For_Agents/blob/main/copy_of_evals_for_agents_datacamp_complete_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evals for Agents with Arize

**DataCamp Code-Along — March 10, 2026**

In this notebook we'll:

1. **Set up tracing** — instrument a Claude agent with Phoenix
2. **Build an agent** — a financial analysis chatbot with web search
3. **Write a code eval** — a deterministic ticker-mention check
4. **Run a built-in eval** — correctness scoring out of the box
5. **Write a custom eval rubric** — an actionability eval from scratch
6. **Run an experiment** — improve the agent and measure the difference

You'll need:
- An Anthropic API key
- A Phoenix Cloud account (free at [app.phoenix.arize.com](https://app.phoenix.arize.com))
  - A Phoenix API key (generate one in Settings)
  - Your Phoenix cloud hostname (looks like https://app.phoenix.arize.com/s/youruser)

## Step 1: Set Up Tracing

Tracing captures a structured record of every step your agent takes — every LLM call, every tool invocation, every decision — with inputs and outputs at each point. We set it up once, and data flows automatically.

In [ ]:
%pip install claude-agent-sdk openinference-instrumentation-claude-agent-sdk arize-phoenix anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.2/309.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.4/180.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.2/432.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.

In [ ]:
import os
from google.colab import userdata

anthropic_api_key = userdata.get('anthropic-api-key')
phoenix_api_key = userdata.get('phoenix-api-key')
phoenix_endpoint = userdata.get('phoenix-collector-endpoint')

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key
os.environ["PHOENIX_API_KEY"] = phoenix_api_key
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="datacamp-claude-financial-agent", auto_instrument=True)

/usr/local/lib/python3.12/dist-packages/phoenix/otel/otel.py:433: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: datacamp-claude-financial-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/lvoss/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Step 2: Build the Agent

We're building a financial analysis chatbot using the Claude Agent SDK. The agent works in two turns:
- **Turn 1: Research** — searches the web for real financial data
- **Turn 2: Write** — compiles the research into a readable report

The `ClaudeSDKClient` maintains conversation context between turns, so the writer has full access to the researcher's findings.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends."""

WRITE_PROMPT = """Now write a concise financial report based on your research above
And output it. Just print it, no need to write to disk."""

options = ClaudeAgentOptions(
    model="claude-haiku-4-5-20251001",
    allowed_tools=["WebSearch"],
    permission_mode="acceptEdits",
)


async def financial_report(tickers: str, focus: str, verbose: bool = True) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research
            if verbose:
                print(f"--- Researching {tickers} ({focus}) ---")
            await client.query(RESEARCH_PROMPT.format(tickers=tickers, focus=focus))
            async for message in client.receive_response():
                if verbose and isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            preview = block.text[:200].replace("\n", " ")
                            print(f"  [research] {preview}...")

            # Turn 2: Write report
            if verbose:
                print(f"--- Writing report ---")
            await client.query(WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                            if verbose:
                                preview = block.text[:200].replace("\n", " ")
                                print(f"  [writing] {preview}...")
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
            return report

### Run the agent once to test

This will take a minute or two

In [ ]:
result = await financial_report("TSLA", "financial performance and growth outlook")
print(result)

--- Researching TSLA (financial performance and growth outlook) ---
  [research] # Tesla (TSLA) Research Report: Financial Performance & Growth Outlook  ## Financial Performance Summary  ### Recent Earnings (Q4 2025) - **EPS**: $0.50 (beat expectations of $0.45) - **Revenue**: ~$2...
--- Writing report ---
  [writing] I'll provide the concise financial report here:  ---  # TESLA, INC. (TSLA) - FINANCIAL REPORT ## March 2026  ---  ## EXECUTIVE SUMMARY  Tesla is at a critical inflection point in 2026, transitioning f...
I'll provide the concise financial report here:

---

# TESLA, INC. (TSLA) - FINANCIAL REPORT
## March 2026

---

## EXECUTIVE SUMMARY

Tesla is at a critical inflection point in 2026, transitioning from traditional automotive growth to an AI and energy-focused strategy. While Q4 2025 results beat expectations, the core EV business faces headwinds. The company is investing heavily in future growth ($20B+ capex in 2026), with energy storage emerging as a strong profit driv

## Step 3: Generate Test Data

To run meaningful evals, we need more than one trace. Here are 12 test queries covering different tickers, focus areas, and levels of complexity.

In [ ]:
test_queries = [
    {"tickers": "AAPL", "focus": "revenue growth and services segment"},
    {"tickers": "NVDA", "focus": "AI chip demand and valuation metrics"},
    {"tickers": "AMZN", "focus": "AWS performance and profitability"},
    {"tickers": "GOOGL", "focus": "advertising revenue and AI strategy"},
    {"tickers": "MSFT", "focus": "cloud computing segment"},
    {"tickers": "META", "focus": "metaverse investments and ad revenue"},
    {"tickers": "TSLA", "focus": "vehicle deliveries and margins"},
    {"tickers": "RIVN", "focus": "financial health and future growth"},
    {"tickers": "AAPL, MSFT", "focus": "comparative financial analysis"},
    {"tickers": "NVDA", "focus": "competitive landscape and market share"},
    {"tickers": "KO", "focus": "dividend yield and stability"},
    {"tickers": "AMZN", "focus": "profitability trends and outlook"},
]

In [ ]:
for i, query in enumerate(test_queries):
    print(f"[{i+1}/{len(test_queries)}] {query['tickers']} — {query['focus']}")
    await financial_report(query["tickers"], query["focus"], verbose=False)
    print(f"  done")

[1/12] AAPL — revenue growth and services segment
  done
[2/12] NVDA — AI chip demand and valuation metrics
  done
[3/12] AMZN — AWS performance and profitability
  done
[4/12] GOOGL — advertising revenue and AI strategy
  done
[5/12] MSFT — cloud computing segment
  done
[6/12] META — metaverse investments and ad revenue
  done
[7/12] TSLA — vehicle deliveries and margins
  done
[8/12] RIVN — financial health and future growth
  done
[9/12] AAPL, MSFT — comparative financial analysis
  done
[10/12] NVDA — competitive landscape and market share
  done
[11/12] KO — dividend yield and stability
  done
[12/12] AMZN — profitability trends and outlook
  done


## Step 4: Code Eval — Ticker Mention Check

Code evals are deterministic functions — no LLM, no API call, no cost. Our simplest useful check: does the output actually mention the ticker we asked about?


In [ ]:
from phoenix.client import Client
import json

px_client = Client()
spans_df = px_client.spans.get_spans_dataframe(project_name="datacamp-claude-financial-agent")

# Filter to top-level agent spans
parent_spans = spans_df[spans_df["parent_id"].isna()].copy()
parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
print(f"Found {len(parent_spans)} top-level spans")

Found 13 top-level spans


In [ ]:
import re
from phoenix.evals import create_evaluator
from phoenix.trace import suppress_tracing
from phoenix.evals import evaluate_dataframe


@create_evaluator(name="mentions_ticker", kind="code")
def mentions_ticker(input, output):
    """Code eval: does the output mention the ticker(s) we asked about?"""
    # Extract ticker symbols from the input (uppercase 1-5 letter words)
    tickers = re.findall(r"\b([A-Z]{1,5})\b", input)
    # Filter to likely tickers (skip common words)
    likely_tickers = [
        t
        for t in tickers
        if len(t) >= 2
        and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
    ]

    if not likely_tickers or not output:
        return {"label": "unknown", "score": 0}

    missing = [t for t in likely_tickers if t not in output.upper()]

    if not missing:
        return {"label": "pass", "score": 1}
    else:
        return {
            "label": "fail",
            "score": 0,
            "explanation": f"Missing tickers: {', '.join(missing)}",
        }


with suppress_tracing():
    results = evaluate_dataframe(dataframe=parent_spans, evaluators=[mentions_ticker])

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
display(results.head(5))

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.llm.token_count.prompt,output,attributes.llm.output_messages,input,attributes.llm.model_name,attributes.tool,attributes.tool.parameters,attributes.tool.name,mentions_ticker_execution_details,mentions_ticker_score
context.span_id,,,,,,,,,,,,,,,,,,,,,
e8545078a862f7ef,financial_report,UNKNOWN,None,2026-03-06 19:37:44.548361+00:00,2026-03-06 19:38:33.530792+00:00,UNSET,,[],e8545078a862f7ef,6612190d2c8f87a8b4044ce6dd0daf9a,...,NaN,"# AMAZON.COM, INC. (AMZN) - FINANCIAL REPORT\n...",None,Research: AMZN\nFocus: profitability trends an...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'mentions_ticker', 'score': 1, 'label..."
faf15253c3330bb4,financial_report,UNKNOWN,None,2026-03-06 19:37:01.927721+00:00,2026-03-06 19:37:44.507526+00:00,UNSET,,[],faf15253c3330bb4,e88602eb73d7ef1658629399a69361e2,...,NaN,# THE COCA-COLA COMPANY (KO)\n## CONCISE FINAN...,None,Research: KO\nFocus: dividend yield and stability,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'mentions_ticker', 'score': 1, 'label..."
3361875fe4ab14d7,financial_report,UNKNOWN,None,2026-03-06 19:36:01.448415+00:00,2026-03-06 19:37:01.886917+00:00,UNSET,,[],3361875fe4ab14d7,36c61e8af377312fbcbf9d974c54c068,...,NaN,```\n═════════════════════════════════════════...,None,Research: NVDA\nFocus: competitive landscape a...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'mentions_ticker', 'score': 1, 'label..."
21d284cbf8220ff8,financial_report,UNKNOWN,None,2026-03-06 19:35:06.461437+00:00,2026-03-06 19:36:01.412521+00:00,UNSET,,[],21d284cbf8220ff8,bd2c437f1cf723a27a9268440cbbe995,...,NaN,# COMPARATIVE FINANCIAL REPORT\n## Apple Inc. ...,None,"Research: AAPL, MSFT\nFocus: comparative finan...",None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'mentions_ticker', 'score': 1, 'label..."
9c7fe33f30da19a5,financial_report,UNKNOWN,None,2026-03-06 19:34:02.911266+00:00,2026-03-06 19:35:06.420028+00:00,UNSET,,[],9c7fe33f30da19a5,05fe01b1f7c423666e5239abee5fa9da,...,NaN,```\n=========================================...,None,Research: RIVN\nFocus: financial health and fu...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'mentions_ticker', 'score': 1, 'label..."


Send the results back to Phoenix

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

# Log mentions_ticker results back to Phoenix
mentions_ticker_evaluations = to_annotation_dataframe(dataframe=results)
Client().spans.log_span_annotations_dataframe(dataframe=mentions_ticker_evaluations)

## Step 5: Built-In Eval — Correctness

Phoenix ships with pre-built evaluators for common checks. The **Correctness** evaluator uses an LLM judge to assess whether a response is factually accurate, complete, and logically consistent — no prompt engineering required.

In [ ]:
from phoenix.evals import LLM
from phoenix.evals.metrics import CorrectnessEvaluator
from phoenix.trace import suppress_tracing
import anthropic

llm = LLM(model="claude-sonnet-4-6", provider="anthropic")

# Built-in evaluator — no prompt template needed
correctness_eval = CorrectnessEvaluator(llm=llm)

We suppress tracing here so it doesn't trace our evaluator itself.

In [ ]:
with suppress_tracing():
    correctness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[correctness_eval]
    )

correctness_results.head()

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.llm.token_count.prompt,output,attributes.llm.output_messages,input,attributes.llm.model_name,attributes.tool,attributes.tool.parameters,attributes.tool.name,correctness_execution_details,correctness_score
context.span_id,,,,,,,,,,,,,,,,,,,,,
e8545078a862f7ef,financial_report,UNKNOWN,None,2026-03-06 19:37:44.548361+00:00,2026-03-06 19:38:33.530792+00:00,UNSET,,[],e8545078a862f7ef,6612190d2c8f87a8b4044ce6dd0daf9a,...,NaN,"# AMAZON.COM, INC. (AMZN) - FINANCIAL REPORT\n...",None,Research: AMZN\nFocus: profitability trends an...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
faf15253c3330bb4,financial_report,UNKNOWN,None,2026-03-06 19:37:01.927721+00:00,2026-03-06 19:37:44.507526+00:00,UNSET,,[],faf15253c3330bb4,e88602eb73d7ef1658629399a69361e2,...,NaN,# THE COCA-COLA COMPANY (KO)\n## CONCISE FINAN...,None,Research: KO\nFocus: dividend yield and stability,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 1.0, 'label':..."
3361875fe4ab14d7,financial_report,UNKNOWN,None,2026-03-06 19:36:01.448415+00:00,2026-03-06 19:37:01.886917+00:00,UNSET,,[],3361875fe4ab14d7,36c61e8af377312fbcbf9d974c54c068,...,NaN,```\n═════════════════════════════════════════...,None,Research: NVDA\nFocus: competitive landscape a...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
21d284cbf8220ff8,financial_report,UNKNOWN,None,2026-03-06 19:35:06.461437+00:00,2026-03-06 19:36:01.412521+00:00,UNSET,,[],21d284cbf8220ff8,bd2c437f1cf723a27a9268440cbbe995,...,NaN,# COMPARATIVE FINANCIAL REPORT\n## Apple Inc. ...,None,"Research: AAPL, MSFT\nFocus: comparative finan...",None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
9c7fe33f30da19a5,financial_report,UNKNOWN,None,2026-03-06 19:34:02.911266+00:00,2026-03-06 19:35:06.420028+00:00,UNSET,,[],9c7fe33f30da19a5,05fe01b1f7c423666e5239abee5fa9da,...,NaN,```\n=========================================...,None,Research: RIVN\nFocus: financial health and fu...,None,None,None,None,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."


And send the results to Phoenix again.

In [ ]:
from phoenix.evals.utils import to_annotation_dataframe

correctness_annotations = to_annotation_dataframe(dataframe=correctness_results)
Client().spans.log_span_annotations_dataframe(dataframe=correctness_annotations)

## Step 6: Custom Eval Rubric — Actionability

Built-in evals are general-purpose. Custom rubrics check what matters for *your* application. Our financial analyst should produce actionable recommendations — not just summarize data, but tell the reader what to do.

A good eval prompt has five parts:
1. **Define the judge's role** — domain context
2. **Explicit criteria** — what passes, what fails
3. **Present the data clearly** — labels and delimiters
4. **Add labeled examples** — show, don't just tell
5. **Constrain the output** — binary labels, not numeric scales

In [ ]:
actionability_template = """
You are an expert financial analyst evaluator. Your task is to judge whether
a financial report provides actionable investment guidance, not just raw data.

ACTIONABLE — The report:
- Contains specific recommendations (buy/sell/hold or equivalent guidance)
- Identifies concrete risks with supporting data
- Includes forward-looking analysis, not just historical data
- Provides context for WHY recommendations are made

NOT ACTIONABLE — The report:
- Only summarizes publicly available data without interpretation
- Lacks specific recommendations or next steps
- Presents risks without supporting evidence
- Contains only backward-looking analysis

Here are examples of each:

Example — ACTIONABLE:
\"Based on NVDA's 122% YoY revenue growth driven by data center demand,
strong forward P/E of 35x relative to sector median of 22x, and expanding
margins, NVDA presents a compelling growth position. Key risk: concentration
in AI training chips (~70% of revenue). Recommendation: accumulate on
pullbacks below $800.\"

Example — NOT ACTIONABLE:
\"NVDA is a major player in the semiconductor industry. The company has seen
significant growth in recent years driven by AI demand. NVDA's stock has
performed well. Investors should consider various factors when making
investment decisions.\"

[BEGIN DATA]
************
User query: {input}
************
Financial Report: {output}
************
[END DATA]

Based on the criteria above, is this financial report ACTIONABLE or NOT ACTIONABLE?
"""

Run our new custom judge.

In [ ]:
from phoenix.evals import ClassificationEvaluator

actionability_evaluator = ClassificationEvaluator(
    name="actionability",
    prompt_template=actionability_template,
    llm=llm,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

with suppress_tracing():
    action_results_df = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[actionability_evaluator]
    )

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
import pandas as pd

pd.json_normalize(action_results_df["actionability_score"].head())

,name,score,label,explanation,kind,direction,metadata.model
0,actionability,1.0,actionable,The report meets the criteria for actionable i...,llm,maximize,claude-sonnet-4-6
1,actionability,0.0,not actionable,The report provides substantial context and an...,llm,maximize,claude-sonnet-4-6
2,actionability,1.0,actionable,This report meets the criteria for actionable ...,llm,maximize,claude-sonnet-4-6
3,actionability,1.0,actionable,The report meets the criteria for actionable i...,llm,maximize,claude-sonnet-4-6
4,actionability,1.0,actionable,The report meets all four criteria for being a...,llm,maximize,claude-sonnet-4-6


Log actionability results back to Phoenix


In [ ]:
action_evaluations = to_annotation_dataframe(dataframe=action_results_df)
Client().spans.log_span_annotations_dataframe(dataframe=action_evaluations)

## Step 7: Improve the Agent and Run an Experiment

Evals told us what's wrong. Now we fix it — and *prove* it's better by running the same inputs through the improved agent and comparing scores side by side.

**The process:**
1. Save failing traces as a dataset (do this in the Phoenix UI)
2. Update the agent's prompts based on eval feedback
3. Run an experiment comparing old vs. new

In [ ]:
# Updated prompts — explicitly request the things our evals found missing

IMPROVED_RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends.

You MUST include:
- Specific financial ratios (P/E, P/B, debt-to-equity, ROE)
- News from the last 6 months
- Current stock price or recent performance data
- Competitive context and market positioning"""

IMPROVED_WRITE_PROMPT = """Now write a concise financial report based on your research above.

The report MUST be actionable. Specifically:
- Include explicit buy/sell/hold recommendations with supporting reasoning
- Identify concrete risks with supporting data
- Include forward-looking analysis, not just historical data
- Provide context for WHY each recommendation is made
- If there are multiple tickers, include a dedicated comparison section"""


async def improved_financial_report(tickers: str, focus: str) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research (improved prompt)
            await client.query(
                IMPROVED_RESEARCH_PROMPT.format(tickers=tickers, focus=focus)
            )
            async for message in client.receive_response():
                pass  # Let the research complete

            # Turn 2: Write report (improved prompt)
            await client.query(IMPROVED_WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
            return report

Create a task - this is what the experiment is testing; in this case, the full agent. But we could be testing an individual tool call or something smaller.

In [ ]:
async def my_task(example):
    # Dataset examples from Phoenix nest the span input: {"input": "Research: TSLA\nFocus: ..."}
    inp = example.input
    if isinstance(inp, dict) and "input" in inp:
        inp = inp["input"]

    # Parse "Research: TSLA\nFocus: vehicle deliveries and margins"
    lines = inp.split("\n")
    tickers = lines[0].replace("Research: ", "").strip()
    focus = lines[1].replace("Focus: ", "").strip() if len(lines) > 1 else ""

    return await improved_financial_report(tickers, focus)

Set up our experiment evaluators to correctly handle the dataset we'll be pulling down.

In [ ]:
from phoenix.evals import ClassificationEvaluator
from phoenix.evals.metrics import CorrectnessEvaluator

# Wrap our evals as experiment-compatible functions
correctness_exp_evaluator = CorrectnessEvaluator(llm=llm)

correctness_exp_evaluator.bind(
    {
        "input": lambda x: x["input"]["input"] if isinstance(x["input"], dict) else x["input"],
        "output": "output",
    }
)

actionability_exp_evaluator = ClassificationEvaluator(
    name="actionability",
    prompt_template=actionability_template,
    llm=llm,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

actionability_exp_evaluator.bind(
    {
        "input": lambda x: x["input"]["input"] if isinstance(x["input"], dict) else x["input"],
        "output": "output",
    }
)

evaluators = [correctness_exp_evaluator, actionability_exp_evaluator]

Now pull down our dataset of failing examples

In [ ]:
# Retrieve the dataset you saved from the Phoenix UI
# (filter traces by failing evals -> Save as Dataset -> name it "datacamp-financial-agent-fails")
dataset = Client().datasets.get_dataset(dataset="datacamp-financial-agent-fails")

display(dataset)

Dataset(id='RGF0YXNldDo0MQ==', name='datacamp-financial-agent-fails', version_id='RGF0YXNldFZlcnNpb246NTc=', examples=5)

And run the experiment!

In [ ]:
from phoenix.client import AsyncClient

async_client = AsyncClient()
experiment = await async_client.experiments.run_experiment(
    dataset=dataset, task=my_task, evaluators=evaluators, timeout=300
)

🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/s/lvoss/datasets/RGF0YXNldDo0MQ==/experiments
🔗 View this experiment: https://app.phoenix.arize.com/s/lvoss/datasets/RGF0YXNldDo0MQ==/compare?experimentId=RXhwZXJpbWVudDoxMDE=


running tasks |          | 0/5 (0.0%) | ⏳ 00:00<? | ?it/s

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

Experiment completed: 5 task runs, 2 evaluator runs, 10 evaluations
